# Module 2: Deploy + Govern

> Self-directed module, ~35 min. Requires LangSmith Deployments permissions to run the deploy step, and Docker for the standalone server section; if you're using LangSmith strictly for observability and evaluations, you can read through without deploying.

Moving the agent to production requires two things:

- **A governance layer** — workspace-level policies that govern every model call (PII and secrets redaction, allow-lists, cost caps). The LangSmith **LLM Gateway** enforces them.
- **Production-ready deployment infrastructure** — a managed runtime for the agent: 30+ endpoints, persistence, HITL, and Studio out of the box. The **Agent Server** runtime is identical whether LangChain operates it (**LangSmith Deployments**) or you do (**standalone server**).

This module wires up the governance layer first, then ships the policy-aware agent both ways — managed in Part 2, self-managed in Part 3.

## Setup


In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / ".env", override=True)

import os

print("LANGSMITH_API_KEY set:        ", bool(os.environ.get("LANGSMITH_API_KEY")))
print("LANGSMITH_API_KEY_GATEWAY set:", bool(os.environ.get("LANGSMITH_API_KEY_GATEWAY")))
print("WORKSPACE_ID set:             ", bool(os.environ.get("WORKSPACE_ID")))
print("ANTHROPIC_API_KEY set:        ", bool(os.environ.get("ANTHROPIC_API_KEY")))

---
# Part 1. Govern — Workspace Policies via the LangSmith Gateway

> The LangSmith LLM Gateway needs to be enabled in your LangSmith org. If it's not visible in your workspace, it needs to be enabled at the organization level, which can be done by your LangSmith administrator. Gateway is currently available on cloud; self-hosted LangSmith support is coming.

The **LangSmith LLM Gateway** is a managed proxy that sits between any client and the model provider (Anthropic, OpenAI, Bedrock, …). Point a model client at the gateway's `base_url` and every request flows through it — no SDK changes needed.

**Gateway policies** are the controls applied at the gateway. They're declared once at the workspace level and enforced on every call, regardless of which agent or notebook is making it.

We'll wire up a policy that redacts PII and secrets, prove it catches a bad prompt, then promote the gateway from a smoke test to the default model config — so both this notebook and the agent we're about to deploy inherit it.

### 1.1 Create a PII / secrets redaction policy

The gateway policy is a workspace-level resource — one POST to `/v1/platform/gateway-policies` and every model call made by every member of the workspace is subject to it.

The payload below declares:

- `action=block` — intercept matched content before forwarding upstream
- `subject_matchers` — apply to this workspace's traffic
- `config.detect.pii / .secrets` — what to look for

In [ ]:
import requests

BASE_URL = "https://api.smith.langchain.com"
api_key = os.environ["LANGSMITH_API_KEY"]
workspace_id = os.environ["WORKSPACE_ID"]

policy = {
    "name": "redact-pii-secrets",
    "description": "Workspace-level PII and secrets redaction",
    "policy_type": "guard",
    "action": "block",
    "subject_matchers": [{"key": "workspace_id", "value": workspace_id}],
    "config": {"version": 1, "detect": {"pii": True, "secrets": True}},
    "enabled": True,
}

resp = requests.post(
    f"{BASE_URL}/v1/platform/gateway-policies",
    headers={"X-Api-Key": api_key, "Content-Type": "application/json"},
    json=policy,
    timeout=30,
)
resp.raise_for_status()
created = resp.json()
print("Policy created.")
print("  id:     ", created.get("id"))
print("  name:   ", created.get("name"))
print("  action: ", created.get("action"))
print("  enabled:", created.get("enabled"))

### 1.2 Route a minimal agent through the gateway

The policy is live workspace-wide, but it only applies to traffic that *goes through the gateway*. To opt a client in, swap its `base_url` to `https://gateway.smith.langchain.com/<provider>`. No other code changes.

Below: a bare `create_deep_agent` whose model client is gateway-routed. We'll exercise it with a clean prompt first to confirm normal traffic still flows.

In [ ]:
from deepagents import create_deep_agent
from langchain.chat_models import init_chat_model

gateway_model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    base_url="https://gateway.smith.langchain.com/openai",
    api_key=os.environ["LANGSMITH_API_KEY_GATEWAY"],
)

demo_agent = create_deep_agent(
    model=gateway_model,
    system_prompt=(
        "You are a helpful assistant behind the LangSmith Gateway, which redacts "
        "sensitive data before you see it, replacing each value with a token like "
        "[SAFE_TO_USE:EMAIL_ADDRESS_a1b2c3]. Treat each token as the real value, "
        "answer normally, and repeat tokens verbatim — the Gateway restores the real "
        "value in your reply. When a reply involves redacted data, start with one line: "
        "🔒 Gateway redacted N sensitive value(s) before I saw them."
    ),
)

ok = demo_agent.invoke({
    "messages": [{"role": "user", "content": "In one sentence, what is the LangSmith LLM Gateway?"}]
})
print(ok["messages"][-1].content)

### 1.3 Watch the policy in action

Same agent, same gateway, but the prompt below carries a (fake) email address. The gateway inspects the request, matches the `guard` policy, and redacts the PII before forwarding upstream — the model sees a `[SAFE_TO_USE:EMAIL_ADDRESS_…]` token, not the real value.

Because of the system prompt from §1.2, the agent flags the redaction in its reply (`🔒 Gateway redacted N sensitive value(s)`). To see exactly what was removed, open this run's trace in LangSmith: the request payload the gateway sent upstream shows the email replaced with a token.

In [ ]:
pii_prompt = (
    "Contact me at jane.doe@example.com for the quarterly report."
)

result = demo_agent.invoke({
    "messages": [{"role": "user", "content": pii_prompt}]
})
print(result["messages"][-1].content)

### 1.4 Make the deployable agent gateway-aware

The demo agent above is provisional. To make every notebook *and* the agent we're about to deploy inherit the gateway, we flip the module's central model config — one edit in `utils/models.py`.

Before (the current default):
```python
# utils/models.py
model = init_chat_model("openai:gpt-4.1-mini")
```

After (gateway-routed):
```python
# utils/models.py
model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    base_url="https://gateway.smith.langchain.com/openai",
    api_key=os.environ["LANGSMITH_API_KEY_GATEWAY"],
)
```

`utils/models.py` already has the gateway block in place, commented out. Comment out the direct line and uncomment the gateway block to flip the switch.

> **Why `LANGSMITH_API_KEY_GATEWAY` and not `LANGSMITH_API_KEY`?**
> `LANGSMITH_API_KEY` is a reserved env var that `langgraph deploy` strips during upload (the platform manages tracing internally). The deployed graph still needs a key for gateway auth, so we read it from a non-reserved name. Both env vars hold the same LangSmith key value — only the name changes.

> **To re-run the other modules later:** revert the edit (the file's comment header explains the toggle). The deployable agent in `agents/deployable_agents/data_agent/` imports `model` from `utils.models`, so whichever block is active at deploy time is what ships.

The cell below reloads the module so this notebook picks up the change without a kernel restart.

In [ ]:
import importlib, utils.models
importlib.reload(utils.models)
from utils.models import model

print("Reloaded utils.models.")
print("  model class:", model.__class__.__name__)
# Anthropic clients expose base_url on the underlying client; print whichever attribute is present.
for attr in ("anthropic_api_url", "openai_api_base", "base_url"):
    value = getattr(model, attr, None) or getattr(getattr(model, "client", None), attr, None)
    if value:
        print(f"  base_url:   ", value)
        break

---
# Part 2. Deploy — Ship the (Now-Governed) Agent

Policies are in place workspace-wide and the module's default model client routes through the gateway. Next, ship the deployment itself.

We deploy the agent in `agents/deployable_agents/data_agent/` to **LangSmith Deployments** with the `langgraph` CLI. Because `agents/deployable_agents/data_agent/agent.py` imports `model` from `utils.models`, the deployed image picks up the gateway `base_url` automatically — no extra flags, no separate config.

## 2.1 Project structure

A deployable LangGraph project is a directory with a `langgraph.json` config at the root that points at one or more graph objects. The repo root already has one — it registers the deployable agent at `agents/deployable_agents/data_agent/agent.py`.

In [ ]:
agent_dir = str(project_root / "agents" / "deployable_agents" / "data_agent")
print("agents/deployable_agents/data_agent/")
print("---")

for root, dirs, files in os.walk(agent_dir):
    dirs[:] = [d for d in dirs if d != "__pycache__"]
    level = root.replace(agent_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")


In [ ]:
langgraph_json_path = project_root / "langgraph.json"
with open(langgraph_json_path) as f:
    print("langgraph.json:")
    print(f.read())

agents_path = os.path.join(agent_dir, "AGENTS.md")
with open(agents_path) as f:
    print("AGENTS.md:")
    print(f.read())


The agent itself lives at `agents/deployable_agents/data_agent/agent.py`. The module-level `agent` variable is what gets deployed — `langgraph.json` references it as `"data_agent": "./agents/deployable_agents/data_agent/agent.py:agent"`.


## 2.3 Local development

Three CLI commands you'll use (all run from the repo root, where `langgraph.json` lives). They're prefixed with `uv run` so they execute against this project's virtual environment rather than a global `langgraph` install:

```bash
# Validate langgraph.json (imports each graph, checks deps)
uv run langgraph validate

# Run locally for development (Studio UI + hot reload)
uv run langgraph dev --port 2024

# Deploy to LangSmith
uv run langgraph deploy
```

`langgraph dev` opens the LangGraph Studio UI in your browser — useful to step through tool calls and approve HITL interrupts visually.

**Docs:** [LangGraph CLI reference](https://docs.langchain.com/langsmith/langgraph-cli) · [Deploy on Cloud](https://docs.langchain.com/langsmith/deploy-to-cloud).

## 2.4 Validate the config

`langgraph validate` imports each graph in `langgraph.json` and verifies dependencies resolve — without building Docker or uploading anything. Use it to catch config and import errors before deploying.

In [ ]:
# `cd` and `!` run in a subshell — chain in one line so cwd applies to the langgraph command.
# `uv run` executes the project venv's langgraph CLI (not a global install).
!cd "{project_root}" && uv run langgraph validate


If `langgraph validate` fails, the most common cause is a missing dependency in `pyproject.toml` or an import error in `agents/deployable_agents/data_agent/agent.py`. Fix locally before running `langgraph deploy` — the deploy build will fail the same way, just slower.


## 2.5 Deploy to LangSmith (optional)

Run the cell below to deploy to **LangSmith Deployments**. `langgraph deploy` builds a Docker image (locally if Docker is available, otherwise remotely on LangSmith's builder) and pushes it. Provisioning takes a few minutes.

> The image we're shipping picks up the gateway `base_url` from `utils/models.py` — no extra deploy flags needed. The deployed agent is governed by the policy from §1.1 from the moment it serves its first request.

> Requires a `LANGSMITH_API_KEY` with deployment permissions — a service key (`lsv2_sk_...`), not a personal token. On Apple Silicon, local builds need Docker Buildx; without Docker, the CLI falls back to a remote build automatically.

Useful flags:
- `--name <name>` — deployment name (defaults to the project directory name)
- `--deployment-type prod` — production deployment (default is `dev`)
- `--remote` — force remote build, skip local Docker
- `--no-wait` — return immediately rather than blocking on status

In [ ]:
# Re-run this command to push a new revision; the CLI finds the existing deployment by name.
# Add `--deployment-type prod` for production, or `--remote` to skip local Docker.
# `uv run` executes the project venv's langgraph CLI (not a global install).
!cd "{project_root}" && uv run langgraph deploy --name langsmith-poc-data-agent --no-input


Deploys take a few minutes to provision. While waiting:

- Watch progress in the **Deployments** tab in LangSmith.
- Open `langgraph dev` in a separate terminal to iterate locally against the same `langgraph.json`.
- Tail logs once provisioning starts: `langgraph deploy logs <deployment-id>`.


### Deployment endpoints

When `langgraph deploy` finishes, the CLI prints the deployment's base URL. Two endpoints worth bookmarking:

- **Studio / home:** the base URL itself — linked from the **Deployments** tab in LangSmith. Opens the deployed graph in Studio.

<img src="../images/deployment_home.png" alt="LangSmith deployment home page" style="display: block; margin: 0; width: auto; max-height: 360px; border-radius: 8px;">

- **OpenAPI / Swagger UI:** append `/docs` to the base URL — interactive API reference showing every endpoint the deployment exposes (`/runs`, `/threads`, `/store`, etc.).

<img src="../images/deployment_docs.png" alt="OpenAPI Swagger UI for the deployed agent" style="display: block; margin: 0; width: auto; max-height: 360px; border-radius: 8px;">

The same URLs are available via `langgraph deploy list` if you lose the original output.


## 2.6 What you get with LangSmith Deployments

Once deployed, the agent is reachable through 30+ endpoints — built once, exposed everywhere:

| Capability | What it enables |
|---|---|
| **REST API** | Standard HTTP requests against `/runs`, `/threads`, `/store` |
| **Studio UI** | Visual debugger to step through state, threads, and tool calls |
| **Agent Protocol** | Stream runs and pause for human input |
| **MCP server** | Other agents can call this one as a tool |
| **A2A** | Agent-to-agent calls with handoffs |
| **Persistent Store** | `/memories/` survives restarts and threads via the platform's Store |
| **HITL** | Interrupt and resume from any client |
| **Cron / Scheduled runs** | Trigger the agent on a schedule |

The registered graphs (`data_agent`, `base_research_agent`, and `client_research`) are each reachable individually through these endpoints once the deployment is live.

---
# Part 3. Standalone Server — Self-Managed Deployment

> Requires Docker with Compose running locally, and a `.env` containing `LANGSMITH_API_KEY`. A production standalone deployment also requires a LangSmith license key (`LANGGRAPH_CLOUD_LICENSE_KEY`). Without Docker, read through — `compose.standalone.yml` is the takeaway.

Part 2 shipped to **LangSmith Deployments**, where LangChain operates both the control plane and the data plane. The **standalone server** model runs the same Agent Server runtime entirely inside infrastructure you own: no control plane, your Postgres, your Redis, your registry, your CI/CD.

Standalone servers are production-ready. Kubernetes (via the LangSmith Helm chart) is the recommended option at production scale; Docker and Compose suit development and smaller workloads, which is what we use below.

| | LangSmith Deployments (Part 2) | Standalone server |
|---|---|---|
| Control plane | LangChain-managed | None |
| Agent Server runtime | Identical | Identical |
| Postgres / Redis | Provisioned for you | You provide |
| Deploy mechanism | `langgraph deploy` | Your pipeline — Docker, Compose, or Kubernetes |
| Scaling and revisions | Managed | You own |
| LangSmith tracing | Built in | Optional, decoupled from deployment |

The runtime contract:

| Variable | Purpose |
|---|---|
| `DATABASE_URI` | Postgres — assistants, threads, runs, thread state, long-term memory, and the background task queue |
| `REDIS_URI` | Redis — pub/sub broker for streaming real-time output from background runs |
| `LANGGRAPH_CLOUD_LICENSE_KEY` | LangSmith license key; authenticates once at server startup. Required for production. |
| `LANGSMITH_API_KEY` | LangSmith access for tracing and evaluation. Sufficient on its own for local and development use. |
| `LANGSMITH_ENDPOINT` | Point tracing at a self-hosted LangSmith instance instead of Cloud |

Two operational constraints worth raising with a platform team:

- **Egress to `https://beacon.langchain.com`** is required for license verification and usage reporting, unless you run air-gapped.
- **Do not run standalone servers in serverless environments.** Scale-to-zero risks task loss, and scaling back up is unreliable.

`GET /ok` is the health endpoint — wire it to your readiness probe.

> **Why not `langgraph up`?** The CLI has two local commands: `langgraph dev` for rapid iteration (no Docker, hot reload) and `langgraph up` for production-like validation of the full stack. Both own the orchestration, so neither shows you the artifact or the infrastructure contract. Part 3 builds the image, then runs *that exact image* against services you declare — the shape of a real pipeline.

The three steps below are the whole model: build once, run the artifact against your own dependencies, verify the API surface.

## 3.1 Build the image

`langgraph build` reads `langgraph.json`, generates a Dockerfile against the `langchain/langgraph-api` base image, installs the project's dependencies, and sets `LANGSERVE_GRAPHS` to point at the registered graphs. The output is an ordinary OCI image in your local Docker cache — scan it, sign it, and promote it through your registry with whatever gates you already run for every other service.

`.dockerignore` at the repo root keeps `.env`, `.venv`, and `.git` out of the build context. Secrets reach the container at runtime; they are never written into a layer.

> The first build installs the full dependency tree and takes several minutes. Run this cell early if you're working through the module live — the rest of Part 3 is fast once the image exists.

> To see exactly what goes into the image, `uv run langgraph dockerfile <path>` writes the generated Dockerfile to disk without building.

In [ ]:
# Build the artifact, then confirm it's an ordinary OCI image.
!cd "{project_root}" && uv run langgraph build -t langsmith-poc-data-agent:local
!docker images langsmith-poc-data-agent:local


## 3.2 Run it against your own Postgres and Redis

`compose.standalone.yml` at the repo root declares the three services a standalone deployment needs. Every value in it maps to infrastructure your platform team already operates, which is what makes it the useful artifact to hand them.

```yaml
# abbreviated — see compose.standalone.yml for the full file
services:
  postgres:
    image: postgres:16
    environment:
      POSTGRES_PASSWORD: ${POSTGRES_PASSWORD:-postgres}
    volumes: [postgres-data:/var/lib/postgresql/data]
    healthcheck: { test: pg_isready -U postgres, interval: 5s, retries: 5 }

  redis:
    image: redis:7
    healthcheck: { test: redis-cli ping, interval: 5s, retries: 5 }

  agent:
    image: ${IMAGE_NAME:-langsmith-poc-data-agent:local}
    ports: ["8123:8000"]
    depends_on:
      postgres: { condition: service_healthy }
      redis:    { condition: service_healthy }
    env_file: [{ path: .env, required: false }]
    environment:
      DATABASE_URI: postgres://postgres:${POSTGRES_PASSWORD:-postgres}@postgres:5432/postgres?sslmode=disable
      REDIS_URI: redis://redis:6379
```

Three details worth pointing out:

- **`env_file`** injects credentials at runtime. Nothing sensitive is in the image, so the artifact is safe to push to a shared registry. If you enabled the gateway in Part 1, `LANGSMITH_API_KEY_GATEWAY` arrives the same way — the standalone container is governed by the same workspace policy.
- **`DATABASE_URI` and `REDIS_URI`** are set explicitly in `environment` because they must resolve to the Compose service names, overriding anything in `.env`.
- **`8123:8000`** — the Agent Server listens on `8000` inside the container. Host port `8123` is arbitrary.

> **The container's environment comes from `.env` alone — it does not inherit your shell.** A model provider key that works in the notebooks because it's exported in your shell will be missing inside the container. The Agent Server constructs the model client while importing each graph, so a missing key fails startup rather than failing on the first request. If the server exits instead of reporting healthy, confirm that the key `utils/models.py` requires is present in `.env`, then read the logs:
>
> ```bash
> docker compose -f compose.standalone.yml logs agent
> ```

In a real pipeline, `IMAGE_NAME` is a registry tag (`registry.internal/agent:$SHA`) rather than a local one, and Compose is Kubernetes. The contract does not change.

In [ ]:
import time
import requests

BASE = "http://localhost:8123"

!cd "{project_root}" && docker compose -f compose.standalone.yml up -d

# First boot applies database migrations, so poll rather than assuming readiness.
for attempt in range(1, 91):
    try:
        if requests.get(f"{BASE}/ok", timeout=2).json().get("ok"):
            print(f"\nAgent Server healthy at {BASE} ({attempt}s)")
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    print("\nServer did not report healthy. Inspect the logs:")
    print("  docker compose -f compose.standalone.yml logs agent")


## 3.3 Inspect the API surface

The container serves its own OpenAPI spec at `/openapi.json` and a Swagger UI at `/docs`. Every deployment environment runs the same Agent Server runtime, so this is the surface Part 2's managed deployment exposes as well — only the operator changed.

The cell below summarizes the spec, then embeds the Swagger UI inline. Open `http://localhost:8123/docs` directly in a browser for the full interactive reference.

In [ ]:
from IPython.display import IFrame, display

spec = requests.get(f"{BASE}/openapi.json", timeout=10).json()
paths = sorted(spec["paths"])

groups = {}
for path in paths:
    root = path.strip("/").split("/")[0] or "(root)"
    groups.setdefault(root, []).append(path)

print(f"{spec['info']['title']} v{spec['info'].get('version', '?')}")
print(f"{len(paths)} paths, served by the image you just built:\n")
for root, members in sorted(groups.items()):
    print(f"  {root:<16} {len(members):>2} path(s)")

# Swagger UI, served by the container itself.
display(IFrame(f"{BASE}/docs", width="100%", height=600))


In [ ]:
# 3.4 Tear down. `-v` also drops the Postgres volume.
!cd "{project_root}" && docker compose -f compose.standalone.yml down -v


---
# Part 4. Test the Deployed Agent in Studio

Exercise the agent through the **Studio UI** — linked from the deployment's page in LangSmith, or available directly at `https://smith.langchain.com/studio` against any registered graph. Studio is a chat-style interface that streams responses, surfaces HITL interrupts inline, and persists each conversation as a thread that lands in the trace project.

Running a representative batch of queries here validates that the deployed agent behaves the same way the local one did. The queries also feed additional trace volume into your LangSmith project, useful for the failure-mode discovery in Module 4.

<img src="../images/Studio_testing.png" alt="LangSmith Studio mid-conversation with the deployed data_agent graph" style="width: auto; max-height: 380px; border-radius: 8px;">

### Test query checklist

Open Studio against the `data_agent` graph and run each query in order. Each one targets a distinct trace shape; together they exercise every code path the agent has.

**Dataset lookup only (single tool, one per demo record)**

1. `What does the card-txn-daily dataset track, and who owns it?`
2. `What classification does credit-risk-scores carry, and how do I get access?`
3. `What metrics are in marketing-campaign-perf?`

**Research only (research subagent, no dataset lookup)**

4. `What's the regulatory definition of probability of default? Search once.`
5. `Summarize recent CFPB guidance on overdraft fees.`
6. `Research industry benchmarks for card cost-per-acquisition. Search at most once.`

**Multi-step (dataset lookup + research, longer trajectories)**

7. `Does our pd_score in credit-risk-scores match the regulatory definition of probability of default? Look it up, then research briefly.`
8. `Look up marketing-campaign-perf, then research a benchmark for cost_per_acquisition.`
9. `Write a data brief on card-txn-daily and save it to /card_brief.md.`

**Edge cases (likely to surface failure modes)**

10. `Look up the dataset 'real-time-fraud-alerts' and tell me its owner.`
11. `What was card-txn-daily's exact gross_spend last Tuesday?`
12. `Which dataset will have the most spend next quarter?`


### What to watch for

While running through the checklist:

- **Trajectory** — catalog queries should show one `search_catalog` call in the trace tree. Research-only queries should show one `task` (subagent delegation) and one or more `web_search` calls inside the subagent. Multi-step queries should chain `search_catalog` → `task` → `write_file`.
- **Unknown-dataset handling** — query 10 should report the catalog returned no match and list the known IDs, not fabricate data.
- **Governance surfacing** — query 2 (`credit-risk-scores`) should surface the Restricted / PII classification and the MRM access requirement, not omit it.
- **Speculative refusal** — query 12 asks for a prediction the agent can't responsibly make, and query 11 asks for a figure not in the catalog. The agent's system prompt and AGENTS.md rules should result in a hedged or declined response, not a confident forecast or a fabricated number.

Optional: repeat the checklist a second time to roughly double the trace volume. Module 4's Insights Agent benefits from a larger corpus — this Studio batch seeds the trace project early, and the later modules add more.


## Recap

| Pillar | What | How |
|---|---|---|
| **Govern** | Workspace-level PII / secrets redaction | `POST /v1/platform/gateway-policies` (`action=block`) |
| **Govern** | Route model calls through the gateway | `init_chat_model(..., base_url="https://gateway.smith.langchain.com/openai")` in `utils/models.py` |
| **Deploy** | Deployable graph config | `langgraph.json` at repo root |
| **Deploy** | Agent identity + skills | `AGENTS.md`, `skills/` |
| **Deploy** | Managed — LangSmith Deployments | `langgraph deploy --name langsmith-poc-data-agent` |
| **Deploy** | Self-managed — standalone server | `langgraph build -t …` → `docker compose -f compose.standalone.yml up -d` |

**The production pattern:** one policy, one `base_url` change, one build. Every model call passes through a governed gateway, and the same Agent Server runtime — 30+ endpoints, persistence, HITL, Studio — is available either as a managed deployment or as an OCI image running on infrastructure you own. The governance layer is independent of which one you choose.

**Next:** `03_tracing.ipynb` — generate traces and query them, now that the governed agent is deployed.